# 03 — Train three xG models with MLflow

The MLflow heart of the demo. We train three competing xG models on
`shots_features` — each in its own MLflow run inside one experiment:

| Run | Model | Why include it |
|-----|-------|----------------|
| `lr_baseline` | `LogisticRegression` | Interpretable baseline. Won't capture the interactions. |
| `rf_v1`       | `RandomForestClassifier` | Captures nonlinearities, no tuning needed. |
| `xgb_v1`      | `XGBoost` | Strong default for tabular. Captures interactions; competitive with LR on this synthetic data. |

For each run we log:
- **Params + custom metrics** — log-loss, Brier score, ROC AUC, PR AUC
- **Signature + input example** — required by UC Model Registry + Serving for type enforcement
- **Artifacts** — ROC curve, calibration curve, and (for tree models) feature importance plot
- **Tags** — `model_family` and `intent` so the next notebook can filter

After this notebook, open the MLflow experiment UI and **compare the three runs side-by-side** —
that's a key piece of the live demo.

In [1]:
import os, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
me = w.current_user.me().user_name

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "hockey_xg_mlflow")
EXPERIMENT = os.getenv("MLFLOW_EXPERIMENT_NAME", f"/Users/{me}/hockey-xg-mlflow")
FEATURES   = f"{UC_CATALOG}.{UC_SCHEMA}.shots_features"

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
exp = mlflow.set_experiment(EXPERIMENT)
print(f"Experiment: {exp.name} (id={exp.experiment_id})")

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


2026/05/28 23:59:39 INFO mlflow.tracking.fluent: Experiment with name '/Users/alexander.booth@databricks.com/hockey-xg-mlflow' does not exist. Creating a new experiment.


Experiment: /Users/alexander.booth@databricks.com/hockey-xg-mlflow (id=112419715816526)


## Load + split

We pull the feature table into pandas — 50k rows is small enough that there's
no benefit to doing distributed training, and sklearn/XGBoost want a single
in-memory matrix anyway.

In [2]:
from sklearn.model_selection import train_test_split

FEATURE_COLS = [
    "distance_ft", "angle_deg", "distance_sq", "angle_sq",
    "is_high_danger", "rebound", "rush", "rebound_dist", "rush_dist",
    "period", "time_in_period_sec", "prev_event_sec",
    "st_wrist", "st_snap", "st_slap", "st_backhand", "st_tip", "st_wrap",
    "str_5v5", "str_5v4_pp", "str_4v5_pk", "str_4v4", "str_3v3_ot", "str_6v5_en", "str_5v6",
]
LABEL = "goal"

pdf = spark.table(FEATURES).select(*FEATURE_COLS, LABEL).toPandas()
X = pdf[FEATURE_COLS].astype("float64")
y = pdf[LABEL].astype("int")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}  Positive rate: {y.mean()*100:.2f}%")

Train: 40,000  Test: 10,000  Positive rate: 9.42%


In [3]:
# Wrap each model so pyfunc.predict() returns P(goal) — not class labels.
# This way the same model works identically in batch (notebook 05) and in
# real-time serving (notebook 06) — every caller gets the xG probability.
import mlflow.pyfunc

class XGPredictor(mlflow.pyfunc.PythonModel):
    def __init__(self, classifier):
        self.classifier = classifier
    def predict(self, context, model_input):
        proba = self.classifier.predict_proba(model_input)
        return proba[:, 1]


/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [4]:
from sklearn.metrics import (
    log_loss, roc_auc_score, brier_score_loss, average_precision_score,
    roc_curve
)
from sklearn.calibration import calibration_curve

def log_eval_artifacts(model, X_te, y_te, name):
    proba = model.predict_proba(X_te)[:, 1]

    # Scalar metrics
    metrics = {
        "test_log_loss": log_loss(y_te, proba),
        "test_brier":    brier_score_loss(y_te, proba),
        "test_roc_auc":  roc_auc_score(y_te, proba),
        "test_pr_auc":   average_precision_score(y_te, proba),
    }
    for k, v in metrics.items():
        mlflow.log_metric(k, v)

    # ROC curve artifact
    fpr, tpr, _ = roc_curve(y_te, proba)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f"AUC={metrics['test_roc_auc']:.3f}")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC — {name}")
    ax.legend()
    plt.tight_layout()
    mlflow.log_figure(fig, "roc_curve.png")
    plt.close(fig)

    # Calibration curve artifact — critical for an xG model
    frac_pos, mean_pred = calibration_curve(y_te, proba, n_bins=10, strategy="quantile")
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="perfect")
    ax.plot(mean_pred, frac_pos, marker="o", label=name)
    ax.set_xlabel("Mean predicted P(goal)")
    ax.set_ylabel("Empirical goal rate")
    ax.set_title(f"Calibration — {name}")
    ax.legend()
    plt.tight_layout()
    mlflow.log_figure(fig, "calibration_curve.png")
    plt.close(fig)

    return metrics

## Run 1 — Logistic Regression baseline

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from mlflow.models.signature import infer_signature

mlflow.sklearn.autolog(disable=True)  # we'll log manually for clarity

with mlflow.start_run(run_name="lr_baseline") as run:
    mlflow.set_tag("model_family", "logistic_regression")
    mlflow.set_tag("intent", "baseline")

    lr = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")),
    ])
    lr.fit(X_train, y_train)

    mlflow.log_params({"model": "logistic_regression", "C": 1.0, "max_iter": 1000})
    metrics = log_eval_artifacts(lr, X_test, y_test, "LR")
    print(f"LR metrics: {metrics}")

    sig = infer_signature(X_train.head(50), lr.predict_proba(X_train.head(50))[:, 1])
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=XGPredictor(lr),
        signature=sig,
        input_example=X_train.head(3),
        pip_requirements=["scikit-learn", "pandas", "numpy"],
    )
    LR_RUN_ID = run.info.run_id
print(f"LR run: {LR_RUN_ID}")

2026/05/28 23:59:48 INFO mlflow.pyfunc: Validating input example against model signature


LR metrics: {'test_log_loss': 0.2753282195258404, 'test_brier': 0.0775311459011405, 'test_roc_auc': 0.7469786593498189, 'test_pr_auc': 0.27296853818675915}


/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Uploading artifacts:   0%|          | 0/7 [00:00<?, ?it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:02,  2.20it/s]

...

Uploading artifacts:  71%|███████▏  | 5/7 [00:00<00:00,  2.20it/s]

Uploading artifacts:  86%|████████▌ | 6/7 [00:00<00:00,  2.20it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:00<00:00,  2.20it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:00<00:00, 14.79it/s]

🏃 View run lr_baseline at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715816526/runs/81e2cbee212d4a69b7a6075d7178794d
🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715816526
LR run: 81e2cbee212d4a69b7a6075d7178794d


## Run 2 — Random Forest

In [6]:
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="rf_v1") as run:
    mlflow.set_tag("model_family", "random_forest")
    mlflow.set_tag("intent", "tree_baseline")

    rf = RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=20,
        n_jobs=-1, random_state=42, class_weight=None,
    )
    rf.fit(X_train, y_train)

    mlflow.log_params({"model": "random_forest", "n_estimators": 200, "max_depth": 10, "min_samples_leaf": 20})
    metrics = log_eval_artifacts(rf, X_test, y_test, "RF")
    print(f"RF metrics: {metrics}")

    # Feature importance plot
    imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(6, 8))
    imp.plot.barh(ax=ax, color="#2ca02c")
    ax.set_title("RF feature importance")
    plt.tight_layout()
    mlflow.log_figure(fig, "feature_importance.png")
    plt.close(fig)

    sig = infer_signature(X_train.head(50), rf.predict_proba(X_train.head(50))[:, 1])
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=XGPredictor(rf),
        signature=sig,
        input_example=X_train.head(3),
        pip_requirements=["scikit-learn", "pandas", "numpy"],
    )
    RF_RUN_ID = run.info.run_id
print(f"RF run: {RF_RUN_ID}")

RF metrics: {'test_log_loss': 0.2786533378006131, 'test_brier': 0.07831333921403466, 'test_roc_auc': 0.7387808409968502, 'test_pr_auc': 0.2594706460442082}


2026/05/28 23:59:53 INFO mlflow.pyfunc: Validating input example against model signature


Uploading artifacts:   0%|          | 0/7 [00:00<?, ?it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:01,  3.14it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:01,  3.14it/s]

...

Uploading artifacts:  71%|███████▏  | 5/7 [00:01<00:00,  3.51it/s]

Uploading artifacts:  86%|████████▌ | 6/7 [00:01<00:00,  3.51it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:01<00:00,  3.51it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:01<00:00,  4.87it/s]

🏃 View run rf_v1 at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715816526/runs/2472c68ffad04ed883cb9f70e8f2500b
🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715816526
RF run: 2472c68ffad04ed883cb9f70e8f2500b


## Run 3 — XGBoost

In [7]:
import xgboost as xgb

with mlflow.start_run(run_name="xgb_v1") as run:
    mlflow.set_tag("model_family", "xgboost")
    mlflow.set_tag("intent", "production_candidate")

    pos = (y_train == 1).sum()
    neg = (y_train == 0).sum()
    scale_pos_weight = neg / max(pos, 1)

    xgb_clf = xgb.XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42,
        scale_pos_weight=1.0,  # keep raw probabilities — better calibration
        tree_method="hist",
    )
    xgb_clf.fit(X_train, y_train)

    mlflow.log_params({
        "model": "xgboost", "n_estimators": 400, "max_depth": 5,
        "learning_rate": 0.05, "subsample": 0.9, "colsample_bytree": 0.9,
    })
    metrics = log_eval_artifacts(xgb_clf, X_test, y_test, "XGB")
    print(f"XGB metrics: {metrics}")

    imp = pd.Series(xgb_clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(6, 8))
    imp.plot.barh(ax=ax, color="#d62728")
    ax.set_title("XGBoost feature importance (gain)")
    plt.tight_layout()
    mlflow.log_figure(fig, "feature_importance.png")
    plt.close(fig)

    sig = infer_signature(X_train.head(50), xgb_clf.predict_proba(X_train.head(50))[:, 1])
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=XGPredictor(xgb_clf),
        signature=sig,
        input_example=X_train.head(3),
        pip_requirements=["scikit-learn", "xgboost", "pandas", "numpy"],
    )
    XGB_RUN_ID = run.info.run_id
print(f"XGB run: {XGB_RUN_ID}")

XGB metrics: {'test_log_loss': 0.2784939998854756, 'test_brier': 0.07832750727553782, 'test_roc_auc': 0.7375326921246846, 'test_pr_auc': 0.2592245301979078}


2026/05/28 23:59:58 INFO mlflow.pyfunc: Validating input example against model signature


Uploading artifacts:   0%|          | 0/7 [00:00<?, ?it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:01,  3.41it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:01,  3.41it/s]

...

Uploading artifacts:  71%|███████▏  | 5/7 [00:00<00:00,  6.68it/s]

Uploading artifacts:  86%|████████▌ | 6/7 [00:00<00:00,  6.68it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:00<00:00,  6.68it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:00<00:00,  8.81it/s]

🏃 View run xgb_v1 at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715816526/runs/86a968c309a744c2a3b60522a6381736
🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715816526
XGB run: 86a968c309a744c2a3b60522a6381736


## Compare the three runs

In the MLflow UI you can multi-select these three runs and compare. In code, we
fetch them sorted by `test_log_loss` (lower is better).

In [8]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.test_log_loss ASC"],
    max_results=5,
)
rows = []
for r in runs:
    rows.append({
        "run_name":     r.data.tags.get("mlflow.runName"),
        "family":       r.data.tags.get("model_family"),
        "log_loss":     r.data.metrics.get("test_log_loss"),
        "brier":        r.data.metrics.get("test_brier"),
        "roc_auc":      r.data.metrics.get("test_roc_auc"),
        "pr_auc":       r.data.metrics.get("test_pr_auc"),
        "run_id":       r.info.run_id,
    })
leaderboard = pd.DataFrame(rows)
print(leaderboard.to_string(index=False))

   run_name              family  log_loss    brier  roc_auc   pr_auc                           run_id
lr_baseline logistic_regression  0.275328 0.077531 0.746979 0.272969 81e2cbee212d4a69b7a6075d7178794d
     xgb_v1             xgboost  0.278494 0.078328 0.737533 0.259225 86a968c309a744c2a3b60522a6381736
      rf_v1       random_forest  0.278653 0.078313 0.738781 0.259471 2472c68ffad04ed883cb9f70e8f2500b


**Next:** `04_evaluate_and_register` picks the leader, runs `mlflow.evaluate` on it,
and registers it to Unity Catalog.